In [ ]:
import pandas as pd
import json

In [ ]:
movies = pd.read_csv("movies.csv")
credits = pd.read_csv("credits.csv")

In [ ]:
movies.shape, credits.shape

In [ ]:
movies["genres"][0]

In [ ]:
json.loads(movies["genres"][0])

In [ ]:
movies.columns

In [ ]:
credits.columns

In [ ]:
movie_credits = pd.merge(movies,credits, left_on = "id", right_on = "movie_id")

In [ ]:
movie_credits.shape

In [ ]:
movie_credits.columns

In [ ]:
movie_credits = movie_credits[['movie_id','title_x','genres','keywords','overview','cast', 'crew']]

In [ ]:
movie_credits.head()

In [ ]:
movie_credits.dropna().shape

In [ ]:
movie_credits.dropna(inplace = True)

In [ ]:
movie_credits.shape

In [ ]:
str_lst = movies["genres"][0]

values = json.loads(str_lst)

lst = []

for value in values:
    lst.append(value["name"])

lst  

In [ ]:
def extract_values(str_lst):
    values = json.loads(str_lst)
    return [value["name"]for value in values]

In [ ]:
extract_values(movies["genres"][0])

In [ ]:
movie_credits["genres"] = movie_credits["genres"].apply(extract_values)

In [ ]:
movie_credits["genres"].head()

In [ ]:
movie_credits["keywords"] = movie_credits["keywords"].apply(extract_values)
movie_credits["keywords"].head()

In [ ]:
movie_credits["cast"] = movie_credits["cast"].apply(extract_values).apply(lambda lst:lst[:3])

In [ ]:
movie_credits["cast"].head()

In [ ]:
movie_credits["crew"][0]

In [ ]:
def fetch_director(str_lst):
    values = json.loads(str_lst)
    return [value["name"]for value in values if value["job"] == "Director"]

In [ ]:
fetch_director(movie_credits["crew"][4])

In [ ]:
movie_credits["crew"] = movie_credits["crew"].apply(fetch_director)

In [ ]:
movie_credits["crew"].head()

In [ ]:
movie_credits['overview'][0]

In [ ]:
movie_credits['overview'] = movie_credits["overview"].str.split()
movie_credits['overview'].head()

In [ ]:
movie_credits.head()

In [ ]:
def collapse(lst):
    final_lst = []
    for i in lst:
        final_lst.append(i.replace(" ",""))
    return final_lst  

In [ ]:
collapse(movie_credits["cast"][0])

In [ ]:
movie_credits["cast"] = movie_credits["cast"].apply(collapse)
movie_credits["crew"] = movie_credits["crew"].apply(collapse)
movie_credits["genres"] = movie_credits["genres"].apply(collapse)
movie_credits["keywords"] = movie_credits["keywords"].apply(collapse)

In [ ]:
movie_credits.head()

In [ ]:
movie_credits["tags"] = movie_credits['overview']+movie_credits["cast"]+movie_credits["crew"] +movie_credits["genres"]+movie_credits["keywords"]

In [ ]:
final_df = movie_credits.drop(columns = ['genres', 'keywords', 'overview', 'cast', 'crew'])

In [ ]:
final_df.rename({"title_x":"title"}, axis = 1, inplace = True)

In [ ]:
final_df["tags"][0]

In [ ]:
final_df

In [ ]:
import nltk

In [ ]:
from nltk.stem.porter import PorterStemmer

In [ ]:
ps = PorterStemmer()

In [ ]:
ps.stem("love")

In [ ]:
ps.stem("lovely")

In [ ]:
ps.stem("loving")

In [ ]:
def stem(lst): 
    final_lst = []

    for i in lst: 
        final_lst.append(ps.stem(i))

    return " ".join(final_lst)

In [ ]:
" ".join(stem(final_df["tags"][0]))

In [ ]:
final_df["tags"] = final_df["tags"].apply(stem)

In [ ]:
final_df.head()

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
cv = CountVectorizer(max_features = 5000, stop_words="english")

In [ ]:
vector = cv.fit_transform(final_df["tags"])
vector

In [ ]:
vector = vector.toarray()
vector

In [ ]:
vector.shape

In [ ]:
print(cv.get_stop_words())

In [ ]:
list(cv.get_stop_words())[50:70]

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
similarity = cosine_similarity(vector)

In [ ]:
similarity.shape

# Recommendation

In [ ]:
final_df.reset_index(drop = True, inplace = True)
final_df

In [ ]:
final_df[final_df["title"] == "Iron Man"].index[0]

In [ ]:
final_df.index

In [ ]:
movie_index = final_df[final_df["title"] == "Iron Man"].index[0]

In [ ]:
distances = similarity[movie_index]

In [ ]:
sorted(distances)

In [ ]:
final_df.iloc[13]

In [ ]:
index_list = list(enumerate(distances))

In [ ]:
index_list

In [ ]:
similar_movie = sorted(index_list, reverse=True, key = lambda x: x[1])[1:11]
similar_movie

In [ ]:
for x in [[1, 1999]]: 
    print(x[1])

In [ ]:
print("danial")

In [ ]:
for i, value in similar_movie:
    print(i, value)

In [ ]:
for i, value in similar_movie:
    print(final_df.iloc[i, 1])

In [ ]:
def recommender(movie_name):
    movie_index = final_df[final_df["title"] == movie_name].index[0]
    distances = similarity[movie_index]
    index_list = list(enumerate(distances))
    similar_movie = sorted(index_list, reverse=True, key = lambda x: x[1])[1:11]
    for i, value in similar_movie:
        print(final_df.iloc[i, 1])

In [ ]:
recommender("X-Men Origins: Wolverine")

In [ ]:
recommender("Thor")

In [ ]:
import pickle

In [ ]:
similarity

In [ ]:
final_df

In [ ]:
pickle.dump(final_df, open("movie_list.pkl", 'wb'))
pickle.dump(similarity, open("similarity.pkl", 'wb'))

In [ ]:
# 1. Convert the massive matrix from float64 to float32 to compress the file size
similarity_compressed = similarity.astype('float32')

# 2. Overwrite your old files with the compressed version
import pickle
pickle.dump(final_df, open("movie_list.pkl", 'wb'))
pickle.dump(similarity_compressed, open("similarity.pkl", 'wb'))
